In [ ]:
# Colab setup - run this cell first
import sys, os

if 'google.colab' in sys.modules:
    repo_dir = '/content/CompMathAndAICourse/'
    code_dir = os.path.join(repo_dir, 'code', '03-optimization')
    if not os.path.exists(repo_dir):
        !git clone https://github.com/lruthotto/CompMathAndAICourse.git {repo_dir}
        %pip install -q jax jaxlib pyyaml

    sys.path.insert(0, code_dir)
    os.chdir(code_dir)
    print(f"Running on Colab, working directory: {code_dir}")
else:
    # Local development - find the directory containing training.py
    _notebook_dir = None
    _cwd = os.getcwd()
    
    # Search paths relative to cwd
    _search_paths = [
        _cwd,
        os.path.join(_cwd, 'code', '03-optimization'),
        os.path.join(_cwd, '03-optimization'),
    ]
    
    for _path in _search_paths:
        if os.path.exists(os.path.join(_path, 'training.py')):
            _notebook_dir = _path
            break
    
    if _notebook_dir:
        os.chdir(_notebook_dir)
        if _notebook_dir not in sys.path:
            sys.path.insert(0, _notebook_dir)
        print(f"Running locally from: {_notebook_dir}")
    else:
        print(f"WARNING: Could not find training.py. CWD is {_cwd}")
        print("Please run this notebook from the 03-optimization directory.")

# Optimization for Neural Network Training

**Lecture 3: SA vs SAA, Backpropagation, and Optimization Regimes**

This notebook demonstrates optimization algorithms for training neural networks on a 2D classification problem (MATLAB peaks function). We explore:

1. **Three network regimes**:
   - **Small**: Standard width (32 neurons), significant feature learning
   - **Lazy/NTK**: Wide network (8192 neurons), minimal feature learning, acts like linear predictor
   - **Mean-field**: Wide network with frozen outer layer, neurons behave as "particles"

2. **Multiple optimizers**:
   - **SGD (Vanilla)**: Basic stochastic gradient descent
   - **Adam**: Adaptive learning rates with momentum
   - **TR-GN**: Trust Region Gauss-Newton (second-order, small networks only)

## Learning Objectives

- Understand how network width affects optimization dynamics
- Compare first-order (SGD, Adam) vs second-order (TR-GN) methods
- Observe the regime shift from small to overparameterized networks
- Experiment with hyperparameters and see their effects

## Setup and Imports

In [4]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np
import matplotlib.pyplot as plt

# Local modules
from training import (
    load_config, print_config,
    generate_peaks_data, init_model,
    train, F,
)
from plotting import (
    setup_style, plot_data, plot_decision_boundary,
    plot_convergence, plot_loss_curves, plot_accuracy_curves,
    plot_training_summary, save_training_results,
    get_optimizer_display_name,
)

# Configure matplotlib
setup_style()

# Check JAX backend
print(f"JAX backend: {jax.default_backend()}")
print(f"JAX version: {jax.__version__}")

ModuleNotFoundError: No module named 'training'

## Configuration

Select the **regime** and **optimizer** to experiment with. The notebook will load pre-tuned hyperparameters, but you can modify them before training.

**Available regimes:**
- `"small"`: Width 32, standard training
- `"lazy"`: Width 8192, NTK regime (slower, needs more iterations)
- `"meanfield"`: Width 4096, frozen outer layer

**Available optimizers:**
- `"sgd_vanilla"`: Vanilla SGD
- `"sgd_momentum"`: SGD with momentum
- `"sgd_nesterov"`: SGD with Nesterov acceleration
- `"adam"`: Adam optimizer  
- `"adamw"`: AdamW (decoupled weight decay)
- `"lion"`: Lion optimizer (sign-based momentum)
- `"tr_gn"`: Trust Region Gauss-Newton (only for small/meanfield)

**Save figures:** Set `SAVE_FIGURES = True` to save plots matching the lecture slide style.

In [ ]:
# ============================================
# SELECT REGIME AND OPTIMIZER HERE
# ============================================

REGIME = "small"          # Options: "small", "lazy", "meanfield"
OPTIMIZER = "adam"        # Options: "sgd_vanilla", "sgd_momentum", "sgd_nesterov",
                          #          "adam", "adamw", "lion", "tr_gn"

# Master random seed for reproducibility
MASTER_SEED = 42

# Save figures to disk (set to True to generate PNG files)
SAVE_FIGURES = False
OUTPUT_DIR = "figures"    # Directory to save figures

# ============================================

In [ ]:
# Load pre-tuned configuration
config = load_config(REGIME, OPTIMIZER)

# Display loaded configuration
print_config(config)

### Modify Hyperparameters (Optional)

You can override any configuration value before training. Uncomment and modify the lines below to experiment.

In [ ]:
# Uncomment to modify hyperparameters:

# config["optimizer"]["learning_rate"] = 0.01
# config["optimizer"]["batch_size"] = 32
# config["optimizer"]["max_iter"] = 5000
# config["optimizer"]["lr_decay"] = 0.99

# For SGD with momentum (if using sgd_momentum config):
# config["optimizer"]["momentum"] = 0.9

# For lazy/meanfield regimes:
# config["model"]["layer_sizes"] = [2, 4096, 5]  # Adjust width

print("Final optimizer config:")
print(f"  Type: {config['optimizer']['type']}")
print(f"  Max iter: {config['optimizer'].get('max_iter')}")
if 'learning_rate' in config['optimizer']:
    print(f"  Learning rate: {config['optimizer']['learning_rate']}")

## Data Generation

We use the MATLAB peaks function to create a 5-class classification problem in 2D. The peaks function creates interesting level sets that provide a non-trivial classification task.

In [ ]:
# Generate data with explicit seed
X_train, y_train, X_test, y_test = generate_peaks_data(config, seed=MASTER_SEED)

# Create key for model initialization
key = random.PRNGKey(MASTER_SEED)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Number of classes: {len(jnp.unique(y_train))}")
print(f"Feature dimension: {X_train.shape[1]}")

In [ ]:
# Visualize the dataset
fig = plot_data(
    np.array(X_train), np.array(y_train),
    np.array(X_test), np.array(y_test),
    title="Peaks Classification Dataset"
)
plt.show()

## Model Initialization

Initialize the neural network with the architecture specified in the config. The initialization method depends on the regime:
- **Xavier** (default): Good for standard training
- **NTK**: For lazy regime (LeCun hidden, N(0,1) output)
- **Mean-field**: For mean-field regime (partitioned indicator output)

In [ ]:
# Initialize model
key, model_key = random.split(key)
theta_init = init_model(config, model_key)

# Count parameters
n_params = sum(layer['W'].size + layer['b'].size for layer in theta_init)
layer_sizes = config['model']['layer_sizes']

print(f"Architecture: {' -> '.join(map(str, layer_sizes))}")
print(f"Total parameters: {n_params:,}")
print(f"Hidden width: {layer_sizes[1]}")

if config.get('model', {}).get('init_type'):
    print(f"Initialization: {config['model']['init_type']}")
if config.get('model', {}).get('width_scale', 0) > 0:
    print(f"Output scaling: 1/width^{config['model']['width_scale']}")
if config.get('model', {}).get('mean_field'):
    print("Mean-field mode: outer layer frozen")

## Training

Train the model using the selected optimizer. The training function handles:
- Mini-batch sampling (for SGD/Adam)
- Learning rate scheduling
- Mean-field gradient masking (freezing outer layer)
- Trust region adaptation (for TR-GN)

In [ ]:
# Train the model
result = train(
    config=config,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    theta_init=theta_init,
    verbose=True,
)

## Results Visualization

In [ ]:
# Create model function for plotting
width_scale = config.get('model', {}).get('width_scale', 0.0)
width = config['model']['layer_sizes'][1]
optimizer_type = config['optimizer']['type']
display_name = get_optimizer_display_name(optimizer_type)

def model_fn(X):
    return F(X, result.params, width_scale=width_scale)

# Plot decision boundary
fig = plot_decision_boundary(
    model_fn,
    np.array(X_train), np.array(y_train),
    np.array(X_test), np.array(y_test),
    title=f"{display_name} (width={width}): Decision Boundary",
    figsize=(6, 5),
)
plt.show()

In [ ]:
# Plot convergence curves (loss + accuracy) in slide style
fig = plot_convergence(
    train_losses=result.train_losses,
    test_losses=result.test_losses,
    train_accuracies=result.train_accuracies,
    test_accuracies=result.test_accuracies,
    iterations=result.iterations,
    optimizer_name=display_name,
    width=width,
    figsize=(12, 4),
)
plt.show()

## Summary

In [ ]:
print("="*60)
print("EXPERIMENT SUMMARY")
print("="*60)
print(f"\nRegime: {REGIME}")
print(f"Optimizer: {OPTIMIZER}")
print(f"Architecture: {' -> '.join(map(str, layer_sizes))}")
print(f"Parameters: {n_params:,}")
print(f"\nFinal Results:")
print(f"  Train Loss: {result.final_train_loss:.4f}")
print(f"  Test Loss:  {result.final_test_loss:.4f}")
print(f"  Train Acc:  {result.final_train_acc:.2%}")
print(f"  Test Acc:   {result.final_test_acc:.2%}")
print(f"  Iterations: {result.n_iterations}")

In [ ]:
# Save figures if requested
if SAVE_FIGURES:
    paths = save_training_results(
        model_fn=model_fn,
        result=result,
        X_train=np.array(X_train),
        y_train=np.array(y_train),
        X_test=np.array(X_test),
        y_test=np.array(y_test),
        optimizer_name=optimizer_type,
        width=width,
        output_dir=OUTPUT_DIR,
        prefix=f"{REGIME}_",
    )
    print("\nSaved files:")
    for name, path in paths.items():
        print(f"  {name}: {path}")
else:
    print("\nTo save figures, set SAVE_FIGURES = True in the config cell.")

---

## Suggested Experiments

Try the following to deepen your understanding:

### 1. Compare Regimes
Run the notebook with:
- `REGIME = "small"` + `OPTIMIZER = "adam"`
- `REGIME = "lazy"` + `OPTIMIZER = "adam"`
- `REGIME = "meanfield"` + `OPTIMIZER = "adam"`

Notice how the decision boundary smoothness and training dynamics change.

### 2. First-Order vs Second-Order
Compare:
- `REGIME = "small"` + `OPTIMIZER = "adam"` (1000 iterations)
- `REGIME = "small"` + `OPTIMIZER = "tr_gn"` (30 iterations)

TR-GN converges much faster per iteration but has O(params²) memory cost.

### 3. Hyperparameter Sensitivity
Try modifying:
- Learning rate (try 10x larger and 10x smaller)
- Batch size (8, 32, 128, full batch)
- Learning rate decay (0.99, 0.999, 1.0)

### 4. Overparameterization Effect
For the small regime, try different widths:
```python
config["model"]["layer_sizes"] = [2, 16, 5]   # Underparameterized
config["model"]["layer_sizes"] = [2, 64, 5]   # Slightly overparameterized
config["model"]["layer_sizes"] = [2, 256, 5]  # Highly overparameterized
```

## Key Takeaways

1. **Regime matters**: Network width fundamentally changes optimization behavior
   - Small: Active feature learning, weights change significantly
   - Lazy: Minimal feature learning, acts like kernel method
   - Mean-field: Moderate feature learning, particle dynamics

2. **Second-order methods** (TR-GN) are competitive in small-scale settings but don't scale to large networks due to O(params²) cost

3. **First-order methods** (SGD, Adam) scale to billions of parameters and benefit from overparameterization

4. **Hyperparameter tuning** is essential for first-order methods; second-order methods are more robust to hyperparameter choices